# 🤖 Machine Learning Model: Customer Satisfaction Rating Prediction
This notebook builds an end-to-end Machine Learning pipeline to predict **Customer Satisfaction Ratings** (1 to 5) directly from ticket text and metadata, and exports the trained model for production deployment.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

# Scikit-Learn tools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 1. Load Data & Filter to Closed Tickets
We train on closed tickets where true `Customer Satisfaction Rating` ground-truth labels exist.

In [2]:
df = pd.read_csv('customer_support_tickets.csv')

# Filter for rows where Customer Satisfaction Rating is present
df_clean = df[df['Customer Satisfaction Rating'].notnull()].copy()
df_clean['Satisfaction_Rating'] = df_clean['Customer Satisfaction Rating'].astype(int)

print(f"Total labeled records: {len(df_clean)}")
print("Class counts:")
print(df_clean['Satisfaction_Rating'].value_counts().sort_index())

Total labeled records: 2769
Class counts:
Satisfaction_Rating
1    553
2    549
3    580
4    543
5    544
Name: count, dtype: int64


## 2. Feature Engineering & Selection
We combine `Ticket Subject` and `Ticket Description` into a single unified text feature `Ticket_Text`, along with key categorical and numerical attributes.

In [3]:
# Combine Subject and Description
df_clean['Ticket_Text'] = df_clean['Ticket Subject'].fillna('') + ' ' + df_clean['Ticket Description'].fillna('')

feature_cols = ['Ticket_Text', 'Ticket Type', 'Ticket Priority', 'Ticket Channel', 'Customer Age']
X = df_clean[feature_cols]
y = df_clean['Satisfaction_Rating']

print(f"Features: {feature_cols}")
print(f"Dataset shape: {X.shape}")
X.head(3)

Features: ['Ticket_Text', 'Ticket Type', 'Ticket Priority', 'Ticket Channel', 'Customer Age']
Dataset shape: (2769, 5)


## 3. Train-Test Split (Stratified 80/20)
Using stratified sampling to maintain balanced class distributions across train and test sets.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

Training set: 2215 samples
Testing set:  554 samples


## 4. Build Preprocessing Pipeline
We use `ColumnTransformer` to handle multimodal inputs:
- **Text (`Ticket_Text`):** TF-IDF Vectorizer with unigrams/bigrams and English stop-words removal.
- **Categoricals (`Ticket Type`, `Ticket Priority`, `Ticket Channel`):** One-Hot Encoding.
- **Numericals (`Customer Age`):** Standard Scaling.

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words='english'), 'Ticket_Text'),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Ticket Type', 'Ticket Priority', 'Ticket Channel']),
        ('num', StandardScaler(), ['Customer Age'])
    ]
)
print("Preprocessing pipeline configured successfully.")

Preprocessing pipeline configured successfully.


## 5. Model 1: Logistic Regression Pipeline
Training a multi-class Logistic Regression classifier within an end-to-end scikit-learn Pipeline.

In [6]:
logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

logreg_pipeline.fit(X_train, y_train)
y_pred_logreg = logreg_pipeline.predict(X_test)

print("=== Logistic Regression Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_logreg):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_logreg))

=== Logistic Regression Performance ===
Accuracy: 0.1949

Classification Report:
              precision    recall  f1-score   support

           1       0.17      0.17      0.17       111
           2       0.21      0.21      0.21       110
           3       0.22      0.22      0.22       116
           4       0.21      0.24      0.23       108
           5       0.15      0.13      0.14       109

    accuracy                           0.19       554
   macro avg       0.19      0.19      0.19       554
weighted avg       0.19      0.19      0.19       554



## 6. Model 2: Random Forest Classifier Pipeline
Training an ensemble tree-based model for comparison.

In [7]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("=== Random Forest Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

=== Random Forest Performance ===
Accuracy: 0.2076

Classification Report:
              precision    recall  f1-score   support

           1       0.23      0.24      0.24       111
           2       0.20      0.16      0.18       110
           3       0.24      0.40      0.30       116
           4       0.20      0.16      0.17       108
           5       0.10      0.06      0.08       109

    accuracy                           0.21       554
   macro avg       0.19      0.21      0.19       554
weighted avg       0.19      0.21      0.20       554



## 7. Model Selection & Export
We select the best performing pipeline and export it using `joblib` so it can be loaded directly by backend APIs or dashboards.

In [8]:
os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', 'customer_satisfaction_model.joblib')

# Save the trained Logistic Regression Pipeline
joblib.dump(logreg_pipeline, model_path)
print(f"Model successfully exported to: {model_path}")

Model successfully exported to: models\customer_satisfaction_model.joblib


## 8. Inference Demonstration on a New Incoming Ticket
Demonstrating how the exported model predicts the satisfaction rating and probabilities for a new raw ticket.

In [9]:
# Load the exported model
loaded_model = joblib.load(model_path)

# Simulated incoming support ticket
sample_ticket = pd.DataFrame([{
    'Ticket_Text': 'Account locked out repeatedly during payment. Unhappy with recurring failures.',
    'Ticket Type': 'Technical issue',
    'Ticket Priority': 'High',
    'Ticket Channel': 'Email',
    'Customer Age': 35
}])

predicted_rating = loaded_model.predict(sample_ticket)[0]
predicted_probs = loaded_model.predict_proba(sample_ticket)[0]

print("--- Inference Results ---")
print(f"Predicted Satisfaction Rating: {predicted_rating} / 5")
print("Confidence Probabilities per rating class:")
for rating, prob in zip(loaded_model.classes_, predicted_probs):
    print(f"  Rating {rating}: {prob * 100:.2f}%")

if predicted_rating <= 2:
    print("\n[ALERT] Customer predicted to have low satisfaction / high frustration risk!")
else:
    print("\n[OK] Customer predicted to have acceptable satisfaction.")

--- Inference Results ---
Predicted Satisfaction Rating: 5 / 5
Confidence Probabilities per rating class:
  Rating 1: 23.35%
  Rating 2: 17.37%
  Rating 3: 19.94%
  Rating 4: 9.00%
  Rating 5: 30.34%

[OK] Customer predicted to have acceptable satisfaction.
